In [11]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
from scipy.signal import convolve2d

In [13]:
s = np.load('psf.npz')
psf = s['psf']

def make_map(files, pole='north', q=0, mu_thr=0.1, binning=4):
    nx, ny = 1024, 1024
    xc, yc = 511.5, 511.5
    Rsun = 480


    if pole == 'north':
        view_new = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=q * 3600) ### North pole view
    else:
        view_new = View(nx, ny, xc, yc, Rsun, 90, -90, 0, rsun_arc=q * 3600) ### South pole view

    grid = view_new.grid()
    mean, variance, coverage, coverage2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[1].header.copy()
            data = hdul[1].data.copy()

        data = rebin(data, binning, update_header=header)
        data = convolve2d(data, psf, mode='same', boundary='symm')

        view = View.from_header(header)

        transform = (view_new.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr))
        grid_, alpha = transform(grid)
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 2
        coverage += np.nan_to_num(n)
        coverage2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)


    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    coverage2[coverage2 == 0] = np.nan
    variance[coverage == 0] = np.nan

    return mean, variance, coverage, coverage2, view_new

In [14]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2302/*'))
print(len(files))
print(files[0], files[-1])

316
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_060000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20251006_100000_TAI.3.magnetogram.fits


In [15]:
mean, variance, coverage, coverage2, view = make_map(files[:], pole='north')

/tmp/ipykernel_72710/536670757.py:42: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [22]:
show_data(mean, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [23]:
variance_ = variance * coverage2 / coverage ** 2
mean_ = mean.copy()
mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan

In [18]:
show_data(mean_, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [19]:
show_data(np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=40)

In [24]:
show_data(np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=10)

In [25]:
show_data(np.abs(mean) / np.sqrt(variance_), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)

In [13]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*'))

In [14]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header.copy()
    data = hdul[0].data.copy()

nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 480

view = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=0)

grid = view.grid()
transform = view.to_carrington(correct_mu=True) + ToSpherical()
grid, _ = transform(grid)

grid = ((np.sin(grid[0] * np.pi / 180) + 1) * 719.5, (grid[1] % 360) * 10)

data = interp2d(data, *grid)

In [15]:
show_data(data, view, label=r'$B_{los}$', vmin=-100, vmax=100)

In [57]:
show_data(np.abs(data) / np.sqrt(variance), view, label=r'$B_{los}$', cmap='turbo', vmin=0, vmax=5)